# Tokenizer

In [6]:
sentences = [
    "Me gusta la tortilla de patatas",
    "La tortilla de champiñones es mucho más jugosa"
]

In [8]:
# sentences[0].split()

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [15]:
# tokenizer.word_index
tokenizer.index_word

{1: 'la',
 2: 'tortilla',
 3: 'de',
 4: 'me',
 5: 'gusta',
 6: 'patatas',
 7: 'champiñones',
 8: 'es',
 9: 'mucho',
 10: 'más',
 11: 'jugosa'}

In [16]:
tokenizer.texts_to_sequences(sentences)

[[4, 5, 1, 2, 3, 6], [1, 2, 3, 7, 8, 9, 10, 11]]

In [24]:
# vocabulario

tam_vocab = 100
tokenizer = Tokenizer(num_words=tam_vocab, oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)
tokenized_sentences = tokenizer.texts_to_sequences(sentences)
tokenized_sentences

[[5, 6, 2, 3, 4, 7], [2, 3, 4, 8, 9, 10, 11, 12]]

# Padding

In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

pad_sequences(tokenized_sentences)

array([[ 0,  0,  5,  6,  2,  3,  4,  7],
       [ 2,  3,  4,  8,  9, 10, 11, 12]])

In [31]:
pad_sequences(tokenized_sentences, maxlen=5, truncating='post')
pad_sequences(tokenized_sentences, maxlen=5, truncating='pre')

array([[ 6,  2,  3,  4,  7],
       [ 8,  9, 10, 11, 12]])

In [33]:
pad_sequences(tokenized_sentences, padding='post')
pad_sequences(tokenized_sentences, padding='pre')

array([[ 0,  0,  5,  6,  2,  3,  4,  7],
       [ 2,  3,  4,  8,  9, 10, 11, 12]])

# Ejemplo de clasificación de textos

## Dataset de Amazon

In [84]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_home.csv")
df = df[df.stars.isin([1,5])]
df['good_product'] = (df.stars == 5).astype(int)
df.drop(columns=['product_category', 'review_title', 'stars'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.sample(5)

,review_body,good_product
5438,"La calidad de los plásticos y de la bolsa es impresionante, muy muy recomendable y el precio imbatible.",1
6696,"Tamaño ideal y muy fácil colocación, ya que cada bombilla dispone de un gancho para sujetar en cuerdas, etc. La luz es muy cálida y suficiente para no encender otras luces. Además viene una bombil...",1
4262,"Lo compré para tenerlo por la noche en la habitación y quitar el olor concentrado de toda la noche, sigue estando el olor a cerrado con el y sin el. Lo he devuelto!!!!",0
783,"no cierran, un desastre, se me cayó todo en el congelador",0
5349,"El mantel no es impermeable y no es antimanchas, menudo engaño",0


In [85]:
df.groupby('good_product').size()

good_product
0    5391
1    5243
dtype: int64

In [86]:
len(df)

10634

## Feature engineering

In [89]:
sentences_bad = df[df.good_product==0]['review_body'].to_list()
sentences_good = df[df.good_product==1]['review_body'].to_list()

In [113]:
tam_vocab = 100

tokenizer_bad = Tokenizer(num_words=tam_vocab)
tokenizer_good = Tokenizer(num_words=tam_vocab)

tokenizer_bad.fit_on_texts(sentences_bad)
tokenizer_good.fit_on_texts(sentences_good)

In [123]:
# tokenizer_good.index_word

In [115]:
all_texts = df['review_body'].to_list()
print(len(all_texts))

10634


In [116]:
tokenized_bad = tokenizer_bad.texts_to_sequences(all_texts)

data = []
for tokens in tokenized_bad:
    row = [1 if i in tokens else 0 for i in range(1, tam_vocab+1)]
    data.append(row)

column_names = [f"b{i}" for i in range(1, tam_vocab+1)]
df_bad = pd.DataFrame(data, columns=column_names)
df_bad.head()

,b1,b2,b3,b4,b5,b6,b7,b8,b9,b10,b11,b12,b13,b14,b15,b16,b17,b18,b19,b20,b21,b22,b23,b24,b25,b26,b27,b28,b29,b30,b31,b32,b33,b34,b35,b36,b37,b38,b39,b40,b41,b42,b43,b44,b45,b46,b47,b48,b49,b50,b51,b52,b53,b54,b55,b56,b57,b58,b59,b60,b61,b62,b63,b64,b65,b66,b67,b68,b69,b70,b71,b72,b73,b74,b75,b76,b77,b78,b79,b80,b81,b82,b83,b84,b85,b86,b87,b88,b89,b90,b91,b92,b93,b94,b95,b96,b97,b98,b99,b100
0,0,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,1,0,1,1,0,0,0,1,0,0,0,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [117]:
tokenized_good = tokenizer_good.texts_to_sequences(all_texts)

data = []
for tokens in tokenized_good:
    row = [1 if i in tokens else 0 for i in range(1, tam_vocab+1)]
    data.append(row)

column_names = [f"g{i}" for i in range(1, tam_vocab+1)]
df_good = pd.DataFrame(data, columns=column_names)
df_good.head()

,g1,g2,g3,g4,g5,g6,g7,g8,g9,g10,g11,g12,g13,g14,g15,g16,g17,g18,g19,g20,g21,g22,g23,g24,g25,g26,g27,g28,g29,g30,g31,g32,g33,g34,g35,g36,g37,g38,g39,g40,g41,g42,g43,g44,g45,g46,g47,g48,g49,g50,g51,g52,g53,g54,g55,g56,g57,g58,g59,g60,g61,g62,g63,g64,g65,g66,g67,g68,g69,g70,g71,g72,g73,g74,g75,g76,g77,g78,g79,g80,g81,g82,g83,g84,g85,g86,g87,g88,g89,g90,g91,g92,g93,g94,g95,g96,g97,g98,g99,g100
0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,0,0,1,1,0,1,0,1,1,0,1,0,0,0,1,0,0,1,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [118]:
df_final = pd.merge(df_bad, df_good, left_index=True, right_index=True, how='inner')
df_final['y'] = df.good_product.values
df_final.sample(5)

,b1,b2,b3,b4,b5,b6,b7,b8,b9,b10,b11,b12,b13,b14,b15,b16,b17,b18,b19,b20,b21,b22,b23,b24,b25,b26,b27,b28,b29,b30,b31,b32,b33,b34,b35,b36,b37,b38,b39,b40,b41,b42,b43,b44,b45,b46,b47,b48,b49,b50,b51,b52,b53,b54,b55,b56,b57,b58,b59,b60,...,g42,g43,g44,g45,g46,g47,g48,g49,g50,g51,g52,g53,g54,g55,g56,g57,g58,g59,g60,g61,g62,g63,g64,g65,g66,g67,g68,g69,g70,g71,g72,g73,g74,g75,g76,g77,g78,g79,g80,g81,g82,g83,g84,g85,g86,g87,g88,g89,g90,g91,g92,g93,g94,g95,g96,g97,g98,g99,g100,y
1842,1,1,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1326,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4345,1,1,0,1,0,1,1,0,1,0,0,0,0,0,0,1,1,1,1,0,0,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9256,0,0,1,0,0,1,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1
8327,0,1,0,1,1,1,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [119]:
df_final.shape

(10634, 201)

In [120]:
df_final.isna().sum().sum()

0

## Modelo 

In [121]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

X = df_final.drop(columns='y')
y = df_final['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=99)

model = DecisionTreeClassifier(random_state=99)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.8048895157498824

# TF-IDF

In [131]:
df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_home.csv")
df['review_body'] = df['review_body'].str.replace('[^a-zA-ZñÑáéíóú .,:;]', '', regex=True)
df.head()

,stars,review_body,review_title,product_category
0,1,Jamás me llegó y el vendedor nunca contacto conmigo a pesar de intentarlo veces,Jamás me llegó,home
1,1,Pone que son piezas y la realidad es que es de piezas,Mala descripción,home
2,1,Saltan los plomos al tercer día de uso. A devolver,Saltan los plomos al utilizarla,home
3,1,No me ha gustado de hecho la devolví . Súper pesada para manejar,No merece la pena,home
4,1,"Por más que busque, no le encuentro el agujero para meter las chuches. En las instrucciones dice que viene tapada con una pegatina transparente, pero la mia no la tiene.",No tiene agujero para rellenar la piñata,home


In [133]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

reviews = df.review_body.values
tfidf_matrix = vectorizer.fit_transform(reviews)

In [135]:
tfidf_matrix.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [136]:
tfidf_matrix.shape

(26962, 21821)

In [137]:
vectorizer.get_feature_names_out()

array(['aa', 'aaa', 'abajo', ..., 'útil', 'útiles', 'útlimo'],
      dtype=object)

In [138]:
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
df_tfidf.head()

,aa,aaa,abajo,abalorios,abandonan,abandonando,abanico,abanicos,abarata,abaratar,abarca,abarcaba,abarcan,abarcar,abarcará,abatible,abeja,aber,abertura,aberturas,abia,abierta,abiertas,abierto,abiertos,abismal,abismo,ablanda,ablandando,ablando,abolla,abollada,abollado,abollados,abolladura,abolladuras,abollan,abolle,abomba,abombada,abombado,abombó,abona,abonaba,abonado,abonan,abonar,abonara,abonarme,abonaron,abonarán,abonen,abono,abonodevolucion,abonó,abra,abran,abras,abrasa,abrasador,...,ánimo,ántes,árbol,árboles,área,áreas,ármate,áspera,ásperas,áspero,ásperos,ático,átomo,él,émbolo,época,épocas,és,éso,ésta,éstas,éste,ésto,éstos,ética,ético,éxito,ímpetu,índice,ínfima,ínfimo,íntegra,íntegro,íntimo,ña,ñapa,ño,ñor,óleo,óptica,óptima,óptimas,óptimo,óptimos,ósea,ósmosis,óxido,última,últimamente,últimas,último,últimos,única,únicamente,únicas,único,únicos,útil,útiles,útlimo
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [148]:
df_tfidf2 = pd.DataFrame(df_tfidf.sum()).reset_index()
df_tfidf2.columns = ['word', 'value']

df_tfidf2.sort_values('value', ascending=False).head(10)

,word,value
5627,de,1484.840507
16843,que,1424.006849
11845,la,1365.794298
14023,no,1320.382281
7418,el,1259.256078
8311,es,1144.827877
13795,muy,1094.555218
12409,lo,956.908244
18707,se,919.493606
7652,en,893.743945


# StopWords

In [143]:
import spacy

nlp = spacy.load("es_core_news_sm")

In [147]:
stopwords_list = nlp.Defaults.stop_words
# list(stopwords_list)

In [149]:
vectorizer = TfidfVectorizer(stop_words=list(stopwords_list))

reviews = df.review_body.values
tfidf_matrix = vectorizer.fit_transform(reviews)

In [150]:
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
df_tfidf.head()

,aa,aaa,abajo,abalorios,abandonan,abandonando,abanico,abanicos,abarata,abaratar,abarca,abarcaba,abarcan,abarcar,abarcará,abatible,abeja,aber,abertura,aberturas,abia,abierta,abiertas,abierto,abiertos,abismal,abismo,ablanda,ablandando,ablando,abolla,abollada,abollado,abollados,abolladura,abolladuras,abollan,abolle,abomba,abombada,abombado,abombó,abona,abonaba,abonado,abonan,abonar,abonara,abonarme,abonaron,abonarán,abonen,abono,abonodevolucion,abonó,abra,abran,abras,abrasa,abrasador,...,ácido,ágil,álbum,álbumes,ámbito,ámbitos,ándame,ángulo,ángulos,ánimo,ántes,árbol,árboles,área,áreas,ármate,áspera,ásperas,áspero,ásperos,ático,átomo,émbolo,época,épocas,és,éso,ésto,ética,ético,éxito,ímpetu,índice,ínfima,ínfimo,íntegra,íntegro,íntimo,ña,ñapa,ño,ñor,óleo,óptica,óptima,óptimas,óptimo,óptimos,ósea,ósmosis,óxido,últimamente,única,únicamente,únicas,único,únicos,útil,útiles,útlimo
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [151]:
df_tfidf.shape

(26962, 21337)

In [152]:
df_tfidf2 = pd.DataFrame(df_tfidf.sum()).reset_index()
df_tfidf2.columns = ['word', 'value']

df_tfidf2.sort_values('value', ascending=False).head(10)

,word,value
3001,calidad,727.601506
15814,precio,562.274190
16111,producto,522.994718
8391,esperaba,326.412765
12312,luz,310.559344
14995,perfecto,272.693927
9456,foto,270.058433
2575,bonito,269.650649
12086,llegado,264.423631
5347,cumple,253.795408


# Ejercicio no-supervisado (Topic Modelling)

In [153]:
from sklearn.cluster import KMeans

k = 5
kmeans = KMeans(n_clusters=k, random_state=99)

In [157]:
X = vectorizer.fit_transform(reviews)
kmeans.fit(X)
labels = kmeans.labels_

In [158]:
df_kmeans = pd.DataFrame(zip(reviews, labels), columns=['review', 'cluster'])
df_kmeans.head()

,review,cluster
0,Jamás me llegó y el vendedor nunca contacto conmigo a pesar de intentarlo veces,4
1,Pone que son piezas y la realidad es que es de piezas,4
2,Saltan los plomos al tercer día de uso. A devolver,4
3,No me ha gustado de hecho la devolví . Súper pesada para manejar,2
4,"Por más que busque, no le encuentro el agujero para meter las chuches. En las instrucciones dice que viene tapada con una pegatina transparente, pero la mia no la tiene.",4


In [159]:
df_kmeans.groupby('cluster').size()

cluster
0     2128
1      868
2      597
3     1269
4    22100
dtype: int64

In [164]:
df_kmeans[df_kmeans.cluster==4].review

0                                                                                                  Jamás me llegó y el vendedor nunca contacto conmigo a pesar de intentarlo  veces
1                                                                                                                           Pone que son  piezas y la realidad es que es de  piezas
2                                                                                                                                Saltan los plomos al tercer día de uso. A devolver
4         Por más que busque, no le encuentro el agujero para meter las chuches. En las instrucciones dice que viene tapada con una pegatina transparente, pero la mia no la tiene.
5        El material está muy duro, cuesta moldear. Una vez los pekes consiguen hacer una figura, no queda sólida al cabo del tiempo como el slime, por lo que al final se deforma.
                                                                                            ...     

# Regex

In [165]:
import re

In [166]:
email = "username.23@mycompany.es"
invalid = "username.23@mycompany.something.es"
long_line = "my email is username.23@mycompany.es and also other.23@mycompany.es"

In [194]:
patron = "[\w+\.]+@\w+\.\w+"

In [195]:
re.findall(patron, email)

['username.23@mycompany.es']

In [199]:
re.findall(patron, invalid)

['username.23@mycompany.something']

In [185]:
re.findall(patron, long_line)

['username.23@mycompany.es', 'other.23@mycompany.es']

In [190]:
# codigo postal

patron = "\d{5}"

re.findall(patron, "28009")
re.findall(patron, "280096")
re.findall(patron, "2800")
re.findall(patron, "280g96")

[]

In [203]:
# ejercicio
# +34-nnn-nnnnnn, nnn-nnnnnn

def check_spanish_phone(tfn):
    
    patron = "(\+34-)?\d{3}-\d{6}"
    match = re.match(patron, tfn)
    is_phone = match is not None
    print(f"{tfn}: {is_phone}")
    
check_spanish_phone("+34-987-654321")
check_spanish_phone("987-654321")
check_spanish_phone("+0034-987-654321")
check_spanish_phone("+34-987654321")
check_spanish_phone("987-654-321")
check_spanish_phone("34987654321")

+34-987-654321: True
987-654321: True
+0034-987-654321: False
+34-987654321: False
987-654-321: False
34987654321: False
